In [3]:
import pennylane as qml
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# =====================================
# Device
# =====================================

dev = qml.device("default.qubit", wires=3)

# =====================================
# QNode 생성 함수
# =====================================

def create_qnode(n_layers):

    @qml.qnode(dev, interface="torch", diff_method="parameter-shift")
    def circuit(params, eta):

        # Input state: H → RZ(eta)
        qml.Hadamard(wires=0)
        qml.RZ(eta, wires=0)

        # Variational layers
        for l in range(n_layers):

            qml.RY(params[l,0], wires=1)
            qml.CNOT(wires=[1,2])

            qml.RY(params[l,1], wires=2)
            qml.CNOT(wires=[2,1])

            qml.RY(params[l,2], wires=1)

        # Cloning interaction block
        qml.CNOT(wires=[0,1])
        qml.CNOT(wires=[0,2])
        qml.CNOT(wires=[1,0])
        qml.CNOT(wires=[2,0])

        # Projector |ψ><ψ|
        psi = torch.stack([
            torch.tensor(1/np.sqrt(2), dtype=torch.cdouble),
            torch.exp(1j*eta)/np.sqrt(2)
        ])

        projector = torch.outer(psi, torch.conj(psi))

        F_B = qml.expval(qml.Hermitian(projector, wires=1))
        F_E = qml.expval(qml.Hermitian(projector, wires=2))

        return F_B, F_E

    return circuit


# =====================================
# PyTorch Model
# =====================================

class VariationalCloner(nn.Module):

    def __init__(self, n_layers):
        super().__init__()

        self.n_layers = n_layers
        self.params = nn.Parameter(
            2*np.pi*torch.rand(n_layers,3)
        )

        self.qnode = create_qnode(n_layers)

    def forward(self, eta):

        F_B, F_E = self.qnode(self.params, eta)

        cost = (
            (1-F_B)**2
            + (1-F_E)**2
            + (F_B - F_E)**2
        )

        return cost, F_B, F_E


# =====================================
# Training
# =====================================

torch.manual_seed(0)

n_layers = 3
model = VariationalCloner(n_layers)
optimizer = optim.Adam(model.parameters(), lr=0.05)

etas = 2*np.pi*torch.rand(30)

epochs = 200
batch_size = 10

print("\n========== TRAINING START ==========\n")

for epoch in range(epochs):

    batch_idx = torch.randperm(len(etas))[:batch_size]
    batch = etas[batch_idx]

    total_loss = 0
    total_FB = 0
    total_FE = 0

    for eta in batch:
        loss, FB, FE = model(eta)
        total_loss += loss
        total_FB += FB
        total_FE += FE

    total_loss /= batch_size
    total_FB /= batch_size
    total_FE /= batch_size

    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d}")
        print(f"  Loss      : {total_loss.item():.6f}")
        print(f"  Fidelity B: {total_FB.item():.6f}")
        print(f"  Fidelity E: {total_FE.item():.6f}")
        print("")

print("\n========== TRAINING COMPLETE ==========\n")

print("Final parameters:\n", model.params.detach().numpy())


========== TRAINING START ==========

Epoch   0
  Loss      : 0.478855
  Fidelity B: 0.605982
  Fidelity E: 0.500088

Epoch  20
  Loss      : 0.080876
  Fidelity B: 0.770676
  Fidelity E: 0.854335

Epoch  40
  Loss      : 0.048880
  Fidelity B: 0.857018
  Fidelity E: 0.833744

Epoch  60
  Loss      : 0.052025
  Fidelity B: 0.850324
  Fidelity E: 0.834243

Epoch  80
  Loss      : 0.043319
  Fidelity B: 0.850391
  Fidelity E: 0.856346

Epoch 100
  Loss      : 0.042722
  Fidelity B: 0.853487
  Fidelity E: 0.854471

Epoch 120
  Loss      : 0.043166
  Fidelity B: 0.853806
  Fidelity E: 0.852724

Epoch 140
  Loss      : 0.042603
  Fidelity B: 0.856524
  Fidelity E: 0.851750

Epoch 160
  Loss      : 0.043671
  Fidelity B: 0.861407
  Fidelity E: 0.844522

Epoch 180
  Loss      : 0.043169
  Fidelity B: 0.852515
  Fidelity E: 0.854286


========== TRAINING COMPLETE ==========

Final parameters:
 [[3.259126   5.2842345  0.49067324]
 [0.76432514 1.4592774  4.0286427 ]
 [3.1239564  6.1979566  2.50